# 0. Initial exploration

This is an analysis the Human Gut Atlas 'Extended+ 18485 genes' dataset originally containing 1.596.000 cells from Oliver et al. (Nature, 2024). It represents a  single-cell map of parts of the the human gastrointestinal tract across different ages and sexes.

The goal is to apply mrVI batch correction to the data in order to uncover subtle sample-specific biological variations in disease tissue that are otherwise masked by technical noise.

This section is only an exploratory visualisation workflow and can be skipped for the actual analysis which starts with 01_subsetting_and_formatting.
However some questions about the data can be explained by examining it closely. To get a perfect understanding of what is contained in the file, I inspected all the elements of the adata and visualised my results in this notebook.

## 0.1. Getting started
### 0.1.0. Packages and display settings

In [29]:
import numpy as np
import pooch
import scanpy as sc
import pandas as pd
import os
from plotnine import * 
import matplotlib.pyplot as plt

In [30]:
# Plotting options
plt.rcParams.update({
    "font.weight": "bold",
    "xtick.major.size": 5,
    "xtick.major.pad": 7,
    "xtick.labelsize": 15,
    "grid.color": "0.5",
    "grid.linestyle": "-",
    "grid.linewidth": 5,
    "lines.linewidth": 2,
    "lines.color": "g",
})

In [31]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" # display all outputs

# Set display to show up to 100 rows and full string length
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

### 0.1.1. File import and defining paths

In [32]:
# Using Pooch to verify file integrity via hashing
file_path = pooch.retrieve(
    url="https://datasets.cellxgene.cziscience.com/1dcf15ee-c103-4aaa-8b8c-0fc697fcccc8.h5ad",
    known_hash=None,
    fname="cellxgene_intestine_oliver.h5ad",
    path="../data",
    progressbar=True
)

# Data is stored as h5ad file
adata = sc.read_h5ad("../data/cellxgene_intestine_oliver.h5ad")

# Define output dir
graphs_dir = "../outputs/figures/00_initial_exploration/"

adata

AnnData object with n_obs × n_vars = 1596200 × 18370
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'control_vs_disease_colors', 'default_embeddi

### 0.1.2. Subsetting of data

As we are interested in researching IBD in juvenile and adult patients, we subset adata to only include the following categories:

- donor_disease: control, organ donor, CD, UC, pediatric IBD
- exclude fetal samples (organ donor)
- organ_groups: large and small intestine

In [33]:
# subset adata to only include the following patient_disease categories: control, organ donor, CD, UC, pediatric IBD
# also: exclude fetal samples (organ donor)

# Select indices first
# Each condition is wrapped in () and separated by &
keep_cells = (adata.obs["donor_disease"].isin(["control","organ_donor","CD","UC","PIBD"])) & \
             (adata.obs["donor_category"].isin(["control","disease"]))

# Slice without an explicit .copy() if you're just moving to the next step
adata = adata[keep_cells]

KeyboardInterrupt: 

In [ ]:
adata

In [ ]:
# Find normalised data

# Find raw data

## 0.2. Inspection of adata.var (features and annotations)

In [ ]:
print(adata.var.shape) #32383 genes described by 13 columns, now 18370 genes described by 7
print(adata.var.columns.tolist())
print(adata.var_names[:5].tolist()) # -> indices are in ENSEMBL IDs

(18370, 7)
['gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
['ENSG00000177757', 'ENSG00000225880', 'ENSG00000230368', 'ENSG00000187634', 'ENSG00000188976']


In [ ]:
adata.var["gene_symbols"].is_unique
adata.var["gene_symbols"].head(5).tolist() # Seems to be unique HGNC nomenclature, should be set as index

True

['FAM87B', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L']

In [ ]:
adata.var["feature_name"].is_unique
adata.var["feature_name"].head(5).tolist()

mismatch_mask = adata.var["feature_name"] != adata.var["gene_symbols"] # What is the difference to gene_symbols?
mismatches = adata.var[mismatch_mask][["feature_name", "gene_symbols"]]

len(mismatches)
mismatches.head(5)
# Audit of gene metadata: Mismatches detected between 'feature_name' and 'gene_symbols'. This indicates varying annotation versions or the use of legacy aliases (e.g. in feature_name KIAA1522) versus updated HGNC nomenclature (in gene_symbols NHSL3).

True

['FAM87B', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L']

815

,feature_name,gene_symbols
ENSG00000215014,SSU72-AS1,AL645728.1
ENSG00000271806,ENSG00000271806,AL590822.2
ENSG00000234396,ENSG00000234396,AL590822.1
ENSG00000225077,ICMT-DT,LINC00337
ENSG00000157330,CFAP107,C1orf158


In [ ]:
# Create a temporary numeric version of the feature length column
temp_lengths = pd.to_numeric(adata.var["feature_length"], errors='coerce')
temp_lengths.max().tolist() #longest gene 104301 bp
temp_lengths.min().tolist() #shortest gene 100 bp

104301

100

In [ ]:
adata.var["feature_type"].value_counts().to_dict()

{'protein_coding': 16894,
 'lncRNA': 1443,
 'transcribed_unprocessed_pseudogene': 20,
 'transcribed_unitary_pseudogene': 13}

In [ ]:
adata.var["feature_reference"].value_counts().to_dict() # All NCBITaxon:9606
adata.var["feature_is_filtered"].value_counts().to_dict() # All false
adata.var["feature_biotype"].value_counts().to_dict() # All gene

{'NCBITaxon:9606': 18370}

{False: 18370}

{'gene': 18370}

## 0.3. Inspection of adata.obs (sample specific metadata)